# Бейзлайн: поиск по словам и локации

**Задача.** На каждый из 2 452 тестовых запросов (файл `benchmark_queries.parquet`) нужно вернуть 50 объявлений из корпуса. Корпус — это 189 212 объявлений услуг из `benchmark_items.parquet`: только среди них и выбираем ответы. Запрос — это набор полей поиска: текст, локация, фильтры, категория и флаг доставки. Какие объявления выбрали пользователи по тестовым запросам, мы не знаем: это проверяет только лидерборд.

**Бейзлайн** — самое простое решение, без обучаемых моделей. С ним сравниваются все следующие шаги. Объявления ищем по словам запроса с помощью BM25, а потом учитываем, где искал пользователь. Код — `avito.retrieval`, ответ на тестовые запросы строит `scripts/submission/baseline.py`.

**Как работает BM25.** Для каждой пары «запрос — объявление» BM25 считает оценку. Она тем выше, чем больше слов запроса нашлось в объявлении, чем реже эти слова встречаются в корпусе (редкое «кератин» важнее частого «услуг») и чем чаще они повторяются в самом объявлении с поправкой на его длину. Слова сравниваем после обработки из `processing/text.ipynb`: текст переводится в нижний регистр, служебные слова вроде «в» и «для» выбрасываются, а окончания отрезаются, чтобы разные формы слова совпадали. Например, «услуги» и «услугам» превращаются в «услуг».

**Как проверяем.** Ответов на тестовые запросы у нас нет, поэтому качество меряем на валидации — 5 032 проверочных запросах из train. Train — это 497 673 строки «поля поиска → выбранное пользователем объявление». Номера запроса у строк нет: строки с одинаковыми полями поиска склеены в один запрос, и все выбранные в них объявления — правильные объявления этого запроса. Как собраны проверочные запросы и чем они похожи на тестовые, показано в `validation/split.ipynb`. Для каждого проверочного запроса берём 1000 объявлений с наибольшей оценкой и считаем:

- **Recall@50** — доля правильных объявлений запроса, попавших в первые 50, усреднённая по всем запросам. Это метрика задачи. У 4 977 из 5 032 проверочных запросов правильное объявление одно (`validation/split.ipynb`, раздел 4), поэтому Recall@50 почти равен доле запросов, у которых правильное объявление попало в первые 50;
- **Recall@200** и **Recall@1000** — то же для первых 200 и первых 1000 объявлений.

**Зачем смотреть дальше первых 50.** Итоговое решение работает в два этапа. Первый этап — ретривер: быстрый поиск отбирает для запроса несколько сотен кандидатов. Этот бейзлайн — первая версия ретривера. Второй этап — реранкер: модель LightGBM заново упорядочивает кандидатов и берёт первые 50 (`ranking/ranker.ipynb`). Правильное объявление, которое попало в первые 200, но не в первые 50, реранкер ещё может поднять, а объявление, которого среди кандидатов нет, он уже не найдёт. Поэтому Recall@200 и Recall@1000 показывают, сколько правильных объявлений поиск вообще находит и сколько можно выиграть на втором этапе.

In [1]:
import numpy as np
import polars as pl

from avito.ingestion import load as ingested
from avito.metrics.recall import bootstrap_interval, recalls_at_k
from avito.processing import load as processed
from avito.retrieval.bm25 import BM25
from avito.retrieval.location import LocationPrior
from avito.retrieval.search import EPSILON, build_documents, retrieve
from avito.validation.load import load_retriever_fold, load_val_queries

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(220);

Загружаем проверочные запросы и корпус. У каждого запроса есть локация поиска — город, регион или вся страна, которые пользователь выбрал на сайте. У каждого объявления есть своя локация — город, где оказывают услугу. В данных локации — это номера без названий. К словам объявлений из `processing` добавляем локацию объявления: она понадобится в разделе 3.

In [2]:
val = load_val_queries()
relevant = val["relevant_item_ids"].to_list()

corpus = processed.load_corpus()
corpus = corpus.join(ingested.load_corpus(["item_id", "item_location_id"]), on="item_id", maintain_order="left")
item_ids = corpus["item_id"].to_numpy()
item_locations = corpus["item_location_id"]

pl.DataFrame({
    "показатель": ["проверочных запросов", "объявлений в корпусе"],
    "значение": [val.height, corpus.height],
})

показатель,значение
str,i64
"""проверочных запросов""",5032
"""объявлений в корпусе""",189212


Функция оценки: по списку из 1000 найденных объявлений на каждый запрос считает Recall@50, Recall@200 и Recall@1000 и добавляет строку в общую таблицу результатов.

Для Recall@50 считаем ещё и 95% интервал: насколько число могло бы сдвинуться, если бы проверочные запросы были другими. Для этого 1000 раз составляем новый набор из 5 032 запросов, вытягивая их случайно из наших: один запрос может попасть в набор несколько раз, а другой — ни разу. На каждом наборе считаем Recall@50. Интервал — это диапазон, в который попали средние 95% этих 1000 значений (такой способ называется бутстреп). Чем уже интервал, тем точнее число.

In [3]:
results = []
recalls_by_method = {}


def evaluate(name: str, top: np.ndarray) -> pl.DataFrame:
    predicted = item_ids[top].tolist()
    recalls = recalls_at_k(predicted, relevant, 50)
    low, high = bootstrap_interval(recalls)
    recalls_by_method[name] = recalls
    results.append({
        "способ": name,
        "Recall@50": recalls.mean(),
        "95% интервал для Recall@50": f"[{low:.3f}, {high:.3f}]",
        "Recall@200": recalls_at_k(predicted, relevant, 200).mean(),
        "Recall@1000": recalls_at_k(predicted, relevant, 1000).mean(),
    })
    return pl.DataFrame(results)

## 1. Поиск только по заголовку

Сначала ищем слова запроса только в заголовке объявления.

In [4]:
bm25_title = BM25().fit(build_documents(corpus, title_repeats=1, params=False, description=False))
top = retrieve(val, bm25_title, item_locations, k=1000)

evaluate("BM25 по заголовку", top)

способ,Recall@50,95% интервал для Recall@50,Recall@200,Recall@1000
str,f64,str,f64,f64
"""BM25 по заголовку""",0.293323,"""[0.281, 0.306]""",0.452272,0.637884


По одному заголовку Recall@50 = 0.29: в первые 50 попадает только 29% правильных объявлений.

## 2. Поиск по заголовку, параметрам и описанию

У объявления три текстовых поля: заголовок, параметры и описание. Параметры — это поля, которые продавец заполняет по шаблону, например «Вид услуги», «Место оказания услуг», «Тип стоимости». В `processing/text.ipynb` взяли 30 000 пар из train «запрос → выбранное объявление» и посмотрели, где в объявлении находятся слова запроса. Все слова запроса есть в заголовке только у 16% пар, в заголовке и параметрах — у 27%, а в заголовке, параметрах и описании вместе — у 61%. Поэтому ищем по всему объявлению: заголовок, параметры и полное описание склеиваются в один текст. Заголовок при этом повторяем трижды, чтобы его слова весили больше, чем слова из длинного описания.

In [5]:
bm25 = BM25().fit(build_documents(corpus))
top = retrieve(val, bm25, item_locations, k=1000)

evaluate("BM25 по всему объявлению", top)

способ,Recall@50,95% интервал для Recall@50,Recall@200,Recall@1000
str,f64,str,f64,f64
"""BM25 по заголовку""",0.293323,"""[0.281, 0.306]""",0.452272,0.637884
"""BM25 по всему объявлению""",0.470191,"""[0.456, 0.484]""",0.706545,0.886592


Поиск по всему объявлению поднимает Recall@50 с 0.29 до 0.47, а Recall@1000 — с 0.64 до 0.89. Но этого всё ещё мало: BM25 смотрит только на слова и не знает, где искал пользователь.

## 3. Локация

BM25 предлагает объявления со всей страны. А в `ingestion/insights.ipynb` видно, что в 83% строк train выбранное объявление лежит в той же локации, где искали.

**Простой способ: сначала своя локация.** Оценки BM25 делим на максимальную по запросу, чтобы лучшее объявление получало 1, а остальные — от 0 до 1. Объявлениям из локации поиска прибавляем ещё 1. Тогда объявления из своей локации стоят выше остальных, а внутри своей локации и среди остальных порядок задаёт BM25.

In [6]:
top = retrieve(val, bm25, item_locations, same_location_bonus=1.0, k=1000)

evaluate("BM25 + своя локация выше всех", top)

способ,Recall@50,95% интервал для Recall@50,Recall@200,Recall@1000
str,f64,str,f64,f64
"""BM25 по заголовку""",0.293323,"""[0.281, 0.306]""",0.452272,0.637884
"""BM25 по всему объявлению""",0.470191,"""[0.456, 0.484]""",0.706545,0.886592
"""BM25 + своя локация выше всех""",0.816839,"""[0.807, 0.827]""",0.899742,0.954491


Одна только локация поднимает Recall@50 с 0.47 до 0.82 — это самый большой шаг во всём решении.

Простой способ не работает для поиска по региону. Локация объявления — всегда конкретный город, а искать можно и по целому региону, например по «Москве и области». Тогда ни одно объявление не совпадает с локацией поиска, прибавку не получает никто, и объявления снова упорядочены только по словам, по всей стране. Кроме того, простой способ не учитывает, что при поиске по городу часть людей выбирает объявления в пригородах.

**Способ получше: доли выборов по локациям.** Для каждой локации поиска считаем по train, в каких локациях и как часто выбирали объявления. Берём при этом не весь train, а обучение ретривера — 482 161 строку, которые остаются, если убрать из train строки проверочных запросов и 6 487 запросов для обучения реранкера (`validation/split.ipynb`). Проверочные запросы убираем, чтобы доли не «знали» их правильных ответов. Запросы реранкера — потому что реранкер получает эти доли как подсказку и учится на своих запросах: если бы доли были посчитаны по тем же запросам, реранкер научился бы им слишком доверять. Для ответа на тестовые запросы `scripts/submission/baseline.py` считает доли по всему train.

Для поиска по «Москве и области» (номер локации `107620`) доли выглядят так:

In [7]:
item_location_table = ingested.load_items(["item_id", "item_location_id"])
pairs = load_retriever_fold(["search_location_id", "item_id"]).join(item_location_table, on="item_id")
location_prior = LocationPrior().fit(pairs)

location_prior.table.filter(pl.col("search_location_id") == 107620).sort("probability", descending=True).head(5).select(
    pl.col("item_location_id").alias("локация объявления"),
    pl.col("probability").alias("доля выборов при поиске по «Москве и области»"),
)

локация объявления,доля выборов при поиске по «Москве и области»
i64,f64
637640,0.60224
638790,0.029039
637770,0.025334
638570,0.024817
639740,0.021715


В данных у локаций нет названий, только номера. Что `637640` — это Москва, а остальные номера в таблице — города Московской области, узнали по координатам выбранных там объявлений (`ingestion/insights.ipynb`). Значит, при поиске по «Москве и области» 60% выборов приходится на саму Москву.

Для скольких локаций поиска посчитаны доли и у скольких проверочных запросов долей для их локации нет (при поиске в такой локации в обучении ретривера ни разу ничего не выбирали):

In [8]:
pl.DataFrame({
    "показатель": ["локаций поиска, для которых посчитаны доли", "проверочных запросов, для локации которых долей нет"],
    "значение": [len(location_prior.transitions), val.filter(~pl.col("search_location_id").is_in(list(location_prior.transitions))).height],
})

показатель,значение
str,i64
"""локаций поиска, для которых посчитаны доли""",2776
"""проверочных запросов, для локации которых долей нет""",2


Доли посчитаны для 2 776 локаций поиска, и нет их только у 2 проверочных запросов. У таких запросов все объявления получают одинаковую добавку за локацию (формула ниже), и порядок задаёт только BM25.

Доли добавляем к оценке объявления так:

`оценка = BM25 / максимальный BM25 по запросу + alpha · log(доля выборов локации объявления + 0.0003)`

Первое слагаемое — оценка по словам, от 0 до 1. Доля выборов не больше 1, поэтому логарифм почти всегда отрицательный, и второе слагаемое — это штраф: чем реже при таком поиске выбирали объявления из локации объявления, тем штраф больше. Логарифм нужен, чтобы разница между 50% и 5% значила столько же, сколько между 5% и 0.5%. Добавка 0.0003 нужна для локаций, где при таком поиске ни разу не выбирали: логарифм нуля не определён, а так их объявления сильно опускаются, но не выбрасываются совсем. `alpha` — вес штрафа: чем он больше, тем сильнее локация перевешивает совпадение по словам.

Какой штраф получает объявление при alpha = 0.1 в зависимости от доли выборов его локации:

In [9]:
location_shares = [0.6, 0.05, 0.005, 0.0]

pl.DataFrame({
    "доля выборов локации объявления": location_shares,
    "добавка к оценке при alpha = 0.1": [0.1 * np.log(share + EPSILON) for share in location_shares],
})

доля выборов локации объявления,добавка к оценке при alpha = 0.1
f64,f64
0.6,-0.051033
0.05,-0.298975
0.005,-0.524005
0.0,-0.811173


Объявление из локации с долей 0.6 (как Москва при поиске по «Москве и области») теряет 0.05, с долей 5% — 0.30, с долей 0.5% — 0.52, а из локации, где ни разу не выбирали, — 0.81, почти всю шкалу оценки по словам. Пробуем alpha 0.05, 0.1 и 0.2.

In [10]:
for alpha in [0.05, 0.1, 0.2]:
    top = retrieve(val, bm25, item_locations, location_prior=location_prior, alpha=alpha, k=1000)
    evaluate(f"BM25 + доли выборов по локациям, alpha = {alpha}", top)

pl.DataFrame(results)

способ,Recall@50,95% интервал для Recall@50,Recall@200,Recall@1000
str,f64,str,f64,f64
"""BM25 по заголовку""",0.293323,"""[0.281, 0.306]""",0.452272,0.637884
"""BM25 по всему объявлению""",0.470191,"""[0.456, 0.484]""",0.706545,0.886592
"""BM25 + своя локация выше всех""",0.816839,"""[0.807, 0.827]""",0.899742,0.954491
"""BM25 + доли выборов по локациям, alpha = 0.05""",0.844959,"""[0.835, 0.855]""",0.929948,0.972377
"""BM25 + доли выборов по локациям, alpha = 0.1""",0.873211,"""[0.864, 0.882]""",0.93336,0.973072
"""BM25 + доли выборов по локациям, alpha = 0.2""",0.850258,"""[0.840, 0.860]""",0.901365,0.939024


Доли выборов по локациям лучше простого «своя локация выше всех»: Recall@50 0.873 против 0.817 при alpha = 0.1. Лучший alpha — 0.1: при 0.05 Recall@50 равен 0.845, при 0.2 — 0.850. При 0.2 штраф за локацию слишком велик: Recall@1000 падает с 0.973 до 0.939, вероятно, потому что объявления из редких для этого поиска локаций опускаются слишком низко, даже если хорошо совпадают по словам.

alpha выбран по тем же проверочным запросам, по которым считаем итог, но всего из трёх значений, поэтому итог от этого завышен, вероятно, ненамного.

## 4. Фильтры запроса

У части запросов есть фильтры, выбранные на сайте, например «Вид услуги Красота, здоровье». Слова фильтров тоже ищем в объявлении: считаем по ним отдельную оценку BM25 и прибавляем её к оценке по тексту запроса с весом 0.3. Формула теперь такая:

`оценка = (BM25 текста запроса + 0.3 · BM25 текста фильтров) / максимум этой суммы по запросу + 0.1 · log(доля выборов локации объявления + 0.0003)`

Вес меньше 1, потому что фильтр вроде «Красота, здоровье» описывает целый раздел услуг, а текст запроса — конкретную услугу. Другие веса фильтров в этом ноутбуке не пробовали.

In [11]:
top = retrieve(val, bm25, item_locations, location_prior=location_prior, alpha=0.1, filter_weight=0.3, k=1000)

evaluate("BM25 + доли выборов по локациям + фильтры", top)

способ,Recall@50,95% интервал для Recall@50,Recall@200,Recall@1000
str,f64,str,f64,f64
"""BM25 по заголовку""",0.293323,"""[0.281, 0.306]""",0.452272,0.637884
"""BM25 по всему объявлению""",0.470191,"""[0.456, 0.484]""",0.706545,0.886592
"""BM25 + своя локация выше всех""",0.816839,"""[0.807, 0.827]""",0.899742,0.954491
"""BM25 + доли выборов по локациям, alpha = 0.05""",0.844959,"""[0.835, 0.855]""",0.929948,0.972377
"""BM25 + доли выборов по локациям, alpha = 0.1""",0.873211,"""[0.864, 0.882]""",0.93336,0.973072
"""BM25 + доли выборов по локациям, alpha = 0.2""",0.850258,"""[0.840, 0.860]""",0.901365,0.939024
"""BM25 + доли выборов по локациям + фильтры""",0.885334,"""[0.876, 0.894]""",0.94747,0.980227


Фильтры поднимают Recall@50 ещё на 0.012: с 0.873 до 0.885. Дальше такие разницы пишем в процентных пунктах (п.п.): 0.012 — это 1.2 п.п.

## 5. Не случайны ли улучшения

Сравниваем соседние шаги на одних и тех же запросах: для каждого запроса считаем, насколько изменился его Recall@50, и строим бутстреп-интервал для средней разницы. Если интервал целиком выше нуля, улучшение не случайное: его нельзя объяснить тем, какие именно запросы попали в валидацию.

In [12]:
def compare(before: str, after: str) -> dict:
    difference = recalls_by_method[after] - recalls_by_method[before]
    low, high = bootstrap_interval(difference)
    return {"было": before, "стало": after, "разница Recall@50": difference.mean(), "95% интервал разницы": f"[{low:.4f}, {high:.4f}]"}


pl.DataFrame([
    compare("BM25 по заголовку", "BM25 по всему объявлению"),
    compare("BM25 по всему объявлению", "BM25 + своя локация выше всех"),
    compare("BM25 + своя локация выше всех", "BM25 + доли выборов по локациям, alpha = 0.1"),
    compare("BM25 + доли выборов по локациям, alpha = 0.1", "BM25 + доли выборов по локациям + фильтры"),
])

было,стало,разница Recall@50,95% интервал разницы
str,str,f64,str
"""BM25 по заголовку""","""BM25 по всему объявлению""",0.176868,"""[0.1632, 0.1906]"""
"""BM25 по всему объявлению""","""BM25 + своя локация выше всех""",0.346648,"""[0.3314, 0.3616]"""
"""BM25 + своя локация выше всех""","""BM25 + доли выборов по локациям, alpha = 0.1""",0.056373,"""[0.0489, 0.0643]"""
"""BM25 + доли выборов по локациям, alpha = 0.1""","""BM25 + доли выборов по локациям + фильтры""",0.012122,"""[0.0086, 0.0157]"""


Все четыре интервала целиком выше нуля, значит каждый шаг действительно улучшает поиск. Даже самый маленький шаг, фильтры, даёт +0.012 с интервалом [0.0086, 0.0157]. Поэтому в бейзлайн входят все четыре шага.

## 6. На каких запросах бейзлайн ошибается

Делим проверочные запросы на группы и считаем Recall@50 лучшего варианта (BM25 + доли выборов по локациям + фильтры) в каждой группе. Каждая пара строк таблицы делит все 5 032 запроса на две части:

- **текст встречается в train / текст новый** — искали ли такой же текст в train в другом запросе, то есть с другой локацией или другими фильтрами. Train здесь — без строк проверочных запросов;
- **без фильтров / с фильтрами**;
- **в локации поиска есть / нет объявлений корпуса** — лежит ли в месте поиска хотя бы одно объявление корпуса. Нет их прежде всего при поиске по региону или всей стране: объявлений с локацией региона не бывает;
- **все правильные объявления в локации поиска / хотя бы одно правильное объявление в другой локации**;
- **число слов в запросе** — после обработки текста, без служебных слов.

In [13]:
corpus_locations = corpus.select(pl.col("item_location_id").alias("search_location_id")).unique()
corpus_locations = corpus_locations.with_columns(pl.lit(True).alias("location_has_items"))

relevant_locations = val.select("query_id", "search_location_id", pl.col("relevant_item_ids").alias("item_id")).explode("item_id", empty_as_null=True)
relevant_locations = relevant_locations.join(item_location_table, on="item_id")
relevant_locations = relevant_locations.group_by("query_id").agg(
    (pl.col("item_location_id") == pl.col("search_location_id")).all().alias("relevant_in_search_location")
)

strata = val.select("query_id", "seen_text", "search_location_id", "search_infm_params_text", "query_stems")
strata = strata.with_columns(pl.Series("recall", recalls_by_method["BM25 + доли выборов по локациям + фильтры"]))
strata = strata.join(relevant_locations, on="query_id", how="left", maintain_order="left")
strata = strata.join(corpus_locations, on="search_location_id", how="left", maintain_order="left")
strata = strata.with_columns(
    pl.col("location_has_items").fill_null(False),
    pl.col("query_stems").str.split(" ").list.len().alias("words"),
)

groups = {
    "текст встречается в train": pl.col("seen_text"),
    "текст новый": ~pl.col("seen_text"),
    "без фильтров": pl.col("search_infm_params_text") == "",
    "с фильтрами": pl.col("search_infm_params_text") != "",
    "в локации поиска есть объявления корпуса": pl.col("location_has_items"),
    "в локации поиска нет объявлений корпуса (в основном поиск по региону или стране)": ~pl.col("location_has_items"),
    "все правильные объявления в локации поиска": pl.col("relevant_in_search_location"),
    "хотя бы одно правильное объявление в другой локации": ~pl.col("relevant_in_search_location"),
    "запрос из 1 слова": pl.col("words") == 1,
    "запрос из 2 слов": pl.col("words") == 2,
    "запрос из 3 слов": pl.col("words") == 3,
    "запрос из 4 слов и больше": pl.col("words") >= 4,
}

pl.DataFrame([
    {"группа запросов": name, "запросов": strata.filter(condition).height, "Recall@50": strata.filter(condition)["recall"].mean()}
    for name, condition in groups.items()
])

группа запросов,запросов,Recall@50
str,i64,f64
"""текст встречается в train""",1861,0.900233
"""текст новый""",3171,0.87659
"""без фильтров""",2634,0.882435
"""с фильтрами""",2398,0.888518
"""в локации поиска есть объявления корпуса""",4150,0.903936
"""в локации поиска нет объявлений корпуса (в основном поиск по региону или стране)""",882,0.797808
"""все правильные объявления в локации поиска""",3899,0.926349
"""хотя бы одно правильное объявление в другой локации""",1133,0.744189
"""запрос из 1 слова""",192,0.776042


Больше всего бейзлайн ошибается в трёх группах:

- хотя бы одно правильное объявление лежит не в той локации, где искали: 0.74 против 0.93;
- в месте поиска нет объявлений корпуса, то есть искали в основном по региону или всей стране: 0.80 против 0.90;
- запрос из одного слова, например «веломеханик» или «прочистка»: 0.78. Вероятно, одного слова мало, чтобы отличить нужное объявление от похожих, но мы это не проверяли.

Запросы с фильтрами и без них бейзлайн решает почти одинаково (0.889 и 0.882). На знакомых текстах он чуть лучше, чем на новых (0.90 против 0.88), хотя не пользуется тем, что выбирали по этим текстам раньше. Вероятно, знакомые тексты — это более частые и типичные запросы, для которых объявление легче найти по словам; мы это не проверяли.

**Что из этого следует для решения.** Ошибки по локации следующий шаг пытается исправить признаками локации в реранкере, например расстоянием от объявления до центра локации поиска (`ranking/ranker.ipynb`). А чтобы находить объявления, которые подходят по смыслу, но написаны другими словами, добавляем поиск по смыслу. Для него нужен эмбеддер — нейросеть, которая превращает текст в вектор; объявления ищутся по близости их векторов к вектору запроса (`embeddings/models.ipynb`, `ranking/dense.ipynb`).

## 7. Чем валидация отличается от теста

Валидация собрана похожей на тест, но повторяет его не во всём (`validation/split.ipynb`, раздел 6). Здесь считаем два отличия, которые касаются бейзлайна. Первое — доля запросов по всему Авито (`search_category = 0`): такие запросы ищут сразу во всех категориях, а не только в «Услугах». С локацией это не связано. Второе — доля запросов без фильтров.

In [14]:
bench = ingested.load_bench_queries()

pl.DataFrame([
    {
        "набор": name,
        "запросов": queries.height,
        "запросов по всему Авито": queries.filter(pl.col("search_category") == 0).height,
        "доля запросов по всему Авито": (queries["search_category"] == 0).mean(),
        "доля запросов без фильтров": (queries["search_infm_params_text"] == "").mean(),
    }
    for name, queries in [("проверочные запросы", val), ("тестовые запросы", bench)]
])

набор,запросов,запросов по всему Авито,доля запросов по всему Авито,доля запросов без фильтров
str,i64,i64,f64,f64
"""проверочные запросы""",5032,1,0.000199,0.52345
"""тестовые запросы""",2452,222,0.090538,0.630506


9% тестовых запросов ищут по всему Авито, а среди проверочных такой запрос всего один из 5 032, поэтому качество на них у себя не проверить. Запросов без фильтров в тесте 63%, в валидации 52%. Но запросы без фильтров бейзлайн решает почти так же, как с фильтрами (0.882 против 0.889, раздел 6), поэтому это отличие, вероятно, мало влияет на итог.

## Итог

Все варианты вместе:

In [15]:
pl.DataFrame(results)

способ,Recall@50,95% интервал для Recall@50,Recall@200,Recall@1000
str,f64,str,f64,f64
"""BM25 по заголовку""",0.293323,"""[0.281, 0.306]""",0.452272,0.637884
"""BM25 по всему объявлению""",0.470191,"""[0.456, 0.484]""",0.706545,0.886592
"""BM25 + своя локация выше всех""",0.816839,"""[0.807, 0.827]""",0.899742,0.954491
"""BM25 + доли выборов по локациям, alpha = 0.05""",0.844959,"""[0.835, 0.855]""",0.929948,0.972377
"""BM25 + доли выборов по локациям, alpha = 0.1""",0.873211,"""[0.864, 0.882]""",0.93336,0.973072
"""BM25 + доли выборов по локациям, alpha = 0.2""",0.850258,"""[0.840, 0.860]""",0.901365,0.939024
"""BM25 + доли выборов по локациям + фильтры""",0.885334,"""[0.876, 0.894]""",0.94747,0.980227


- **Описание важно.** Поиск по всему объявлению вместо одного заголовка: Recall@50 0.29 → 0.47.
- **Локация — самая сильная подсказка.** Объявления из своей локации выше всех: 0.47 → 0.82. Доли выборов по локациям вместо этого: ещё +5.6 п.п., 0.82 → 0.87. Доли задуманы так, чтобы учитывать поиск по региону и выборы в пригородах; по группам запросов два способа мы здесь не сравнивали. Лучший вес локации alpha = 0.1.
- **Фильтры** как дополнительные слова запроса с весом 0.3: ещё +1.2 п.п. Итог бейзлайна — Recall@50 = 0.885, 95% интервал [0.876, 0.894]. Интервалы всех четырёх улучшений целиком выше нуля.
- **Где ошибается:** хотя бы одно правильное объявление в другой локации (0.74), поиск в основном по региону или стране (0.80), однословные запросы (0.78).
- **Что дальше.** Recall@200 бейзлайна — 0.947, Recall@1000 — 0.980, а Recall@50 — только 0.885. Значит, многие правильные объявления бейзлайн находит, но ставит ниже 50-го места. Их может поднять второй этап — реранкер (`ranking/ranker.ipynb`).

Всё это — на валидации. На лидерборде, то есть на тестовых запросах, бейзлайн получил Recall@50 = 0.825 (таблица в `README.md`). Вероятно, отчасти разрыв объясняется тем, что 9% тестовых запросов ищут по всему Авито, а в валидации такой запрос один (раздел 7), и тем, что запросов с несколькими правильными объявлениями в тесте, судя по условию, больше (`validation/split.ipynb`, раздел 6). Какие причины дают основную часть разрыва, мы не проверяли.